# Tracery

A self-contained refresher on **Tracery** — Kate Compton's tiny library for **generative
text grammars**. You hand it a JSON dictionary of symbols and their possible expansions,
ask it to *flatten* a starting symbol, and it does recursive, randomized find-and-replace
until no placeholders remain. It is the engine behind a generation of Twitterbots
(via *Cheap Bots, Done Quick!*) and a go-to for designer-friendly procedural text.

**Domain:** Procedural Generation  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**Tracery is a replacement grammar.** A grammar is a plain dictionary: each *key* is a
**symbol**, each value is a **list of expansion rules**. You start from one symbol
(conventionally `origin`) and **flatten** it: every `#symbol#` reference is replaced by a
randomly chosen rule, and any `#symbol#`s *inside* that rule are expanded too, recursively,
until the string is all literal text.

**The problem it solves.** You want lots of varied, structured text — bot tweets, NPC
barks, item descriptions, flavor names, prompts — and you want a *non-programmer* (a writer
or designer) to be able to author and tweak it. Tracery's entire grammar is JSON, so the
authoring surface is "edit a dictionary," not "write code." The combinatorics do the heavy
lifting: a handful of lists with 5–10 entries each multiply into thousands of distinct
outputs.

**Reach for it when:**
- You need **combinatorial variety** from **hand-authored** building blocks (Mad-Libs at scale).
- A **writer/designer** should own the content without touching the program.
- Output is **short-to-medium** and structure matters more than long-range meaning.
- You want it **portable**: the same JSON runs in JS (the original), Python, and other ports.

**Look elsewhere when:**
- You need **semantic coherence** or facts over a paragraph — Tracery has no notion of meaning.
- You want output that **learns** from a corpus — that's Markov chains or an LLM, not a grammar.
- You need **grammatical correctness** in a non-English language — the built-in modifiers are
  naive English-only.

## 2. Mental Model

**Mad Libs where every blank can contain more blanks.**

A grammar is a dictionary of symbols → lists of rules. Flattening is recursive
find-and-replace: pick a random rule for the current `#symbol#`, splice it in, then expand
any `#symbol#`s the rule itself introduced. Repeat until no `#` remain.

```
grammar = {
  "origin":  ["#greeting#, brave #hero#!"],
  "greeting":["Hello", "Greetings", "Hey"],
  "hero":    ["knight", "wanderer", "#hero# of the #place#"],   # rules can recurse
  "place":   ["north", "deep wood"]
}

flatten("#origin#")
  #origin#
   └─► "#greeting#, brave #hero#!"          (chose rule 0 of origin)
        ├─ #greeting# ─► "Greetings"          (random pick)
        └─ #hero#     ─► "wanderer"           (random pick)
  = "Greetings, brave wanderer!"
```

Two refinements layered on top of plain replacement:

- **Modifiers** — `#hero.capitalize#` post-processes the chosen text (`.a`, `.s`, `.ed`,
  `.capitalize`, …). They chain left-to-right.
- **Actions** — `[hero:#name#]` *locks in* a random pick under a new key so later references
  reuse the **same** value, giving you consistency (a character that keeps one name).

## 3. Key Concepts

- **Symbol** — a key in the grammar dictionary (e.g. `"hero"`). Referenced in text as
  `#hero#`. When flattened it expands to one randomly chosen rule.
- **Rule** — one entry in a symbol's list of possible expansions. Rules can contain literal
  text *and* further `#symbol#` references (including the symbol itself → recursion).
- **`origin`** — convention for the top-level / entry symbol you flatten. Not special to the
  engine; you can flatten any symbol.
- **`flatten(s)`** — expand a string containing `#symbol#`s down to pure text. Picks are
  random, so each call can differ.
- **Modifiers** — `#symbol.modifier#`. Built-in English set: `capitalize`, `capitalizeAll`,
  `a` (indefinite article a/an), `s` (pluralize), `ed` (past tense), `uppercase`,
  `lowercase`, `replace`. They **chain**: `#animal.a.capitalize#`.
- **Actions / saved variables** — `[key:#rule#]` evaluates `#rule#` once and binds the
  result to `key`, so every later `#key#` reuses it. The mechanism for *consistency* across a
  single flatten (same hero name throughout a sentence/story).
- **Grammar (JSON)** — the whole grammar is a JSON object, which is why it ports trivially
  between languages and tools.
- **Randomness** — Tracery uses Python's global `random`; seed it (`random.seed(...)`) for
  reproducible output.

## 4. Setup

`tracery` is a pure-Python package, no native deps, CPU-only — everything below runs in
milliseconds on any machine.

```bash
%pip install tracery
```

The built-in English modifiers (`.capitalize`, `.a`, `.s`, …) live separately and must be
registered on each grammar:

```python
import tracery
from tracery.modifiers import base_english
g = tracery.Grammar(rules)
g.add_modifiers(base_english)   # without this, .capitalize etc. are no-ops
```

There is no network or API key involved, so the notebook executes top-to-bottom in a fresh
kernel as-is.

In [ ]:
# %pip install tracery   # uncomment if not already installed
import random
import tracery
from tracery.modifiers import base_english

random.seed(7)  # reproducible picks for this notebook
print("tracery imported OK")
print("built-in modifiers:", list(base_english.keys()))

## 5. Worked Examples

### Example 1 — A grammar with references and modifiers

A small "greeter" grammar. Note how `#place.a#` adds the right article (*a*/*an*) and
`#mood.capitalize#` post-processes the chosen word. Each `flatten` re-rolls every symbol.

In [ ]:
rules = {
    "origin":   ["#greeting#, traveler. You've reached #place.a#. #mood.capitalize# tidings."],
    "greeting": ["Hail", "Well met", "Greetings"],
    "place":    ["old observatory", "enchanted forest", "abandoned mill"],
    "mood":     ["fair", "uneasy", "strange"],
}

grammar = tracery.Grammar(rules)
grammar.add_modifiers(base_english)   # enable .a, .capitalize, ...

for _ in range(4):
    print(grammar.flatten("#origin#"))

### Example 2 — Saved variables for consistency (`[key:#rule#]`)

Plain references re-roll independently, so `#name# ... #name#` can disagree. An **action**
`[hero:#name#]` picks a name **once** and binds it; every later `#hero#` reuses it. This is
how you keep a character (or place, or color scheme) consistent across one generated piece.

In [ ]:
story_rules = {
    # [hero:#name#] and [foe:#name#] lock in two distinct-looking picks up front.
    "origin": ["[hero:#name#][foe:#name#]#hero# the #title# set out to face #foe#. "
               "By dusk, #hero# had #outcome#."],
    "name":    ["Ada", "Bram", "Cora", "Idris", "Wren"],
    "title":   ["Bold", "Quick", "Quiet", "Tireless"],
    "outcome": ["triumphed", "turned back", "vanished into the fog"],
}

g = tracery.Grammar(story_rules)
g.add_modifiers(base_english)

for _ in range(4):
    print(g.flatten("#origin#"))

### Example 3 — Authoring as JSON + a custom modifier + recursion

Real Tracery grammars are authored as JSON (e.g. for *Cheap Bots, Done Quick!*), so loading
from a JSON string is the realistic path. This grammar also shows **recursion**
(`adjchain` referencing itself to stack adjectives) and a **custom modifier** registered
alongside the English ones. `flatten_count` does the combinatorial counting for you here by
sampling, illustrating how a few short lists yield large variety.

In [ ]:
import json

grammar_json = """{
  "origin":   ["The #adjchain# #creature# guards the #place.shout#"],
  "adjchain": ["#adj#", "#adj#, #adjchain#"],
  "adj":      ["ancient", "glittering", "restless", "moss-covered"],
  "creature": ["wyrm", "golem", "sphinx", "owl"],
  "place":    ["bridge", "archive", "spring"]
}"""

rules = json.loads(grammar_json)
g = tracery.Grammar(rules)
g.add_modifiers(base_english)

# Custom modifier: registered like the built-ins, called as #symbol.shout#
g.add_modifiers({"shout": lambda text, *params: text.upper() + "!"})

random.seed(1)
samples = {g.flatten("#origin#") for _ in range(200)}
for line in list(samples)[:5]:
    print(line)
print(f"\n200 draws produced {len(samples)} distinct lines from 4 tiny lists + recursion.")

## 6. Gotchas & Pitfalls

- **Forgetting `add_modifiers(base_english)`.** Without it, `#word.capitalize#` doesn't error
  — it flattens to literal `word((.capitalize))`. If modifiers seem ignored, you skipped the
  registration step.
- **Missing symbols fail silently-ish.** A reference to an undefined symbol expands to
  `((symbolname))` rather than raising. Watch for double-parens in output — it means a typo'd
  or absent key.
- **Plain references don't stay consistent.** `#name# and #name#` are two independent rolls.
  Use a saved variable `[n:#name#]#n# and #n#` when you need the same value twice.
- **Unbounded recursion.** A self-referential rule with no terminating branch
  (`"x": ["#x#!"]`) recurses until the stack blows. Always give recursive symbols a
  non-recursive escape rule (see `adjchain` above).
- **Global randomness.** Tracery draws from Python's global `random`. Two grammars share that
  stream; seed `random.seed(...)` once for reproducibility, and don't expect per-grammar
  independence.
- **Modifier order matters.** Chains run left-to-right: `#animal.a.capitalize#` →
  *"An owl"*, but `#animal.capitalize.a#` applies `.a` to already-capitalized text and can
  misjudge the article. Put `.capitalize` last.
- **English-only, naive modifiers.** `.s` and `.ed` use simple rules (good enough for *cat→
  cats*, wrong for *child*/*goose*). For other languages or irregulars, author the correct
  forms as explicit rules instead of relying on modifiers.
- **Escaping literal `#` and `[ ]`.** These characters are syntax. To emit them literally,
  escape with a backslash (`\\#`) or build them from a symbol whose rule is the bare
  character.

## 7. When to Use vs Alternatives

| Approach | Best for | Trade-off vs Tracery |
|---|---|---|
| **Tracery** | Hand-authored, designer-editable combinatorial text; bots, barks, names. | No semantics, no learning; English-only modifiers; coherence is on *you*. |
| **Context-free grammars (raw)** | Same idea, more formal/expressive (weights, full CFG tooling). | More machinery; Tracery is the friendly, JSON-shaped subset. See `context-free-grammars`. |
| **Markov chains** | Mimicking the *style* of an existing corpus you don't want to hand-author. | Learns from data but drifts/incoherent; no structural control. See `markov-chains`. |
| **L-systems** | Recursive *structural* growth (plants, fractals), not prose. | Different goal — geometry/structure, not readable sentences. See `l-systems`. |
| **LLMs** | Fluent, coherent, semantically-aware long text. | Heavy, nondeterministic, costly, hard to constrain to an exact format; overkill for a bot tweet. |
| **Template strings / f-strings** | One-off fills with little variety. | Fine until you want nested choice and recombination — then Tracery is far less code. |

**Rule of thumb:** if a writer can express the content as "here are the slots and here are
the words that go in each slot," Tracery is the lightest tool that scales it. The moment you
need the text to *mean* something coherent across sentences, move up to an LLM (or post-filter
Tracery output through one).

## 8. Resources

- **Official Tracery site & interactive tutorial** — http://tracery.io/
- **`tracery` Python package (PyPI)** — https://pypi.org/project/tracery/
- **Reference JS source (Kate Compton, `galaxykate/tracery`)** — https://github.com/galaxykate/tracery
- **Cheap Bots, Done Quick!** (the Twitterbot host that made Tracery famous) — https://cheapbotsdonequick.com/
- **"Practical Procedural Generation for Everyone"** — Kate Compton's GDC talk on the design
  philosophy behind Tracery — https://www.youtube.com/watch?v=WumyfLEa6bU
- Related notebooks in this domain: `context-free-grammars`, `markov-chains`, `l-systems`.

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def flatten(text, grammar, choose, modifiers=None, depth=0):
    """Recursive randomized replacement, with modifiers and saved variables."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE